# MatchCast AI — GPU Processing on Google Colab (T4)

Run the perception pipeline (YOLOv8 detection → ByteTrack → homography) on a free Colab **T4 GPU**. This is ~50–150× faster than CPU: a full match goes from days to well under an hour (minutes with frame sampling).

## One-time setup (in Google Drive)

1. Enable the GPU: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.
2. In your Google Drive, create a folder named **`MatchCast_Colab`** containing:
   - the **`perception/`** folder (copy it from `d:\Match_Cast\perception`)
   - your model file **`best.pt`** (from `d:\Match_Cast\data\models\best.pt`)
   - your match video, e.g. **`match.mp4`**

Then run the cells below in order. The output `tracking.json` is downloaded at the end — drop it into `d:\Match_Cast\data\outputs\<match_id>\tracking.json` locally and your app will show the full analysis.

### 1. Confirm the GPU is active

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### 2. Install dependencies

In [ ]:
!pip -q install ultralytics supervision opencv-python-headless pydantic

### 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 4. Configure paths (edit these to match your Drive)

In [ ]:
import os

# Folder in your Drive holding perception/, best.pt and the video
BASE_DIR    = '/content/drive/MyDrive/MatchCast_Colab'
MODEL_PATH  = os.path.join(BASE_DIR, 'best.pt')
VIDEO_PATH  = os.path.join(BASE_DIR, 'match.mp4')

MATCH_ID    = 'colab_match_01'   # any id; used for the output folder name
FRAME_STRIDE = 5                 # 1 = every frame. On T4, 3–5 is fast + smooth.
FRAME_LIMIT  = None              # None = whole video, or an int to cap frames
OUTPUT_DIR   = '/content/outputs'

assert os.path.isdir(BASE_DIR), f'Missing folder: {BASE_DIR}'
assert os.path.isfile(MODEL_PATH), f'Missing model: {MODEL_PATH}'
assert os.path.isfile(VIDEO_PATH), f'Missing video: {VIDEO_PATH}'
assert os.path.isdir(os.path.join(BASE_DIR, 'perception')), f'Missing perception/ in {BASE_DIR}'
print('All paths OK.')

### 5. Run the pipeline on GPU

Ultralytics auto-detects the GPU, so no extra config is needed. Progress prints every 50 analyzed frames.

In [ ]:
import sys, time
sys.path.insert(0, BASE_DIR)  # so `import perception` works

from perception.pipeline import PerceptionPipeline

t0 = time.time()
pipeline = PerceptionPipeline(model_path=MODEL_PATH)
out_json = pipeline.process_video(
    video_path=VIDEO_PATH,
    match_id=MATCH_ID,
    output_dir=OUTPUT_DIR,
    limit_frames=FRAME_LIMIT,
    frame_stride=FRAME_STRIDE,
)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print('Output:', out_json)

### 6. Download the result

Also saves a copy back to your Drive folder. Then place `tracking.json` locally at `d:\Match_Cast\data\outputs\<MATCH_ID>\tracking.json`.

In [ ]:
import shutil
from google.colab import files

drive_copy = os.path.join(BASE_DIR, f'tracking_{MATCH_ID}.json')
shutil.copy(out_json, drive_copy)
print('Saved to Drive:', drive_copy)
files.download(out_json)